In [177]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report

# --- LOAD DATA ---
df = pd.read_csv("cleaned_patient_with_hospital.csv")
df_thai = pd.read_csv("patient.csv")

# --- FILTER DISEASE ---
df = df[df["disease_type"].str.lower().str.contains("diarrhea")].copy()
df_thai = df_thai[df_thai['ชื่อกลุ่มโรค'].str.contains("อุจจาระร่วง")]

In [178]:
df_thai["วันที่เริ่มป่วย"].value_counts()

วันที่เริ่มป่วย
12/1/2026    630
5/1/2026     594
14/1/2026    578
26/1/2026    563
16/2/2026    550
            ... 
27/3/2026    176
29/3/2026    126
28/3/2026    121
30/3/2026     62
31/3/2026      1
Name: count, Length: 90, dtype: int64

In [179]:
df_raw = pd.read_csv("cleaned_patient_with_hospital.csv")
df_raw["date_disease_start"] = pd.to_datetime(
    df_raw["date_disease_start"],
    dayfirst=True,
    errors="coerce"
)
print(df_raw["date_disease_start"].dt.month.unique())
print(df_raw["date_disease_start"].dt.year.unique())
# df_thai["วันที่เริ่มป่วย"] = pd.to_datetime(df_thai["วันที่เริ่มป่วย"], errors="coerce")
# print(df_thai["วันที่เริ่มป่วย"].dt.month.unique())
# print(df_thai["วันที่เริ่มป่วย"].dt.year.unique())

[1 2 3]
[2026]


In [180]:
print(df_raw["date_disease_start"].head(10))

0   2026-01-01
1   2026-01-01
2   2026-01-01
3   2026-01-01
4   2026-01-01
5   2026-01-01
6   2026-01-01
7   2026-01-01
8   2026-01-01
9   2026-01-01
Name: date_disease_start, dtype: datetime64[ns]


In [181]:
# --- PARSE DATE ---
df["date_disease_start"] = pd.to_datetime(
    df["date_disease_start"],
    dayfirst=True,
    errors="coerce"
)
df = df.dropna(subset=["date_disease_start"])

# df["year"] = df["date_disease_start"].dt.year
# df["month"] = df["date_disease_start"].dt.month
df["year"] = df["date_disease_start"].dt.year
df["week"] = df["date_disease_start"].dt.isocalendar().week
# --- AGGREGATE ---
# monthly_cases = (
#     df.groupby(["district", "year", "month"])
#     .size()
#     .reset_index(name="case_count")
# )
weekly_cases = (
    df.groupby(["district", "year", "week"])
    .size()
    .reset_index(name="case_count")
)

# Sort properly for lag features
weekly_cases["date"] = pd.to_datetime(
    weekly_cases["year"].astype(str) + "-W" + weekly_cases["week"].astype(str) + "-1",
    format="%G-W%V-%u"   # ISO week format
)
# monthly_cases = monthly_cases.sort_values(by=["district", "year", "month"])
# monthly_cases.head()
weekly_cases = weekly_cases.sort_values(["district", "date"])
weekly_cases.head()

,district,year,week,case_count,date
0,bang_bon,2026,1,38,2025-12-29
1,bang_bon,2026,2,67,2026-01-05
2,bang_bon,2026,3,99,2026-01-12
3,bang_bon,2026,4,81,2026-01-19
4,bang_bon,2026,5,77,2026-01-26


In [182]:
# --- FEATURE ENGINEERING ---
# monthly_cases["lag_1"] = monthly_cases.groupby("district")["case_count"].shift(1)
# monthly_cases["lag_2"] = monthly_cases.groupby("district")["case_count"].shift(2)

# monthly_cases["rolling_avg_2"] = monthly_cases[["lag_1", "lag_2"]].mean(axis=1)
# monthly_cases["rolling_max_2"] = monthly_cases[["lag_1", "lag_2"]].max(axis=1)

# monthly_cases["month_of_year"] = monthly_cases["month"]
weekly_cases["lag_1"] = weekly_cases.groupby("district")["case_count"].shift(1)
weekly_cases["lag_2"] = weekly_cases.groupby("district")["case_count"].shift(2)

weekly_cases["rolling_avg_2"] = weekly_cases[["lag_1", "lag_2"]].mean(axis=1)
weekly_cases["rolling_max_2"] = weekly_cases[["lag_1", "lag_2"]].max(axis=1)

weekly_cases["week_of_year"] = weekly_cases["week"]

# Drop rows with NaN (from lag)
# monthly_cases = monthly_cases.dropna()
# weekly_cases = weekly_cases.dropna()
weekly_cases[["lag_1", "lag_2"]] = weekly_cases[["lag_1", "lag_2"]].fillna(0)

# --- LABEL CREATION ---
# threshold = np.percentile(monthly_cases["case_count"], 75)
# monthly_cases["label"] = (monthly_cases["case_count"] > threshold).astype(int)
# monthly_cases.head()
threshold = np.percentile(weekly_cases["case_count"], 75)
weekly_cases["label"] = (weekly_cases["case_count"] > threshold).astype(int)
weekly_cases.head()

,district,year,week,case_count,date,lag_1,lag_2,rolling_avg_2,rolling_max_2,week_of_year,label
0,bang_bon,2026,1,38,2025-12-29,0.0,0.0,NaN,NaN,1,0
1,bang_bon,2026,2,67,2026-01-05,38.0,0.0,38.0,38.0,2,0
2,bang_bon,2026,3,99,2026-01-12,67.0,38.0,52.5,67.0,3,1
3,bang_bon,2026,4,81,2026-01-19,99.0,67.0,83.0,99.0,4,1
4,bang_bon,2026,5,77,2026-01-26,81.0,99.0,90.0,99.0,5,1


In [183]:
# --- CREATE DATE FOR SPLIT ---
# monthly_cases["date"] = pd.to_datetime(
#     monthly_cases["year"].astype(str) + "-" + monthly_cases["month"].astype(str) + "-01"
# )

# split_index = int(len(monthly_cases) * 0.8)

# train_set = monthly_cases.iloc[:split_index]
# test_set  = monthly_cases.iloc[split_index:]
weekly_cases = weekly_cases.sort_values(['district','date']).reset_index(drop=True)

split_index = int(len(weekly_cases) * 0.8)

train_set = weekly_cases.iloc[:split_index]
test_set  = weekly_cases.iloc[split_index:]
train_set.head()

,district,year,week,case_count,date,lag_1,lag_2,rolling_avg_2,rolling_max_2,week_of_year,label
0,bang_bon,2026,1,38,2025-12-29,0.0,0.0,NaN,NaN,1,0
1,bang_bon,2026,2,67,2026-01-05,38.0,0.0,38.0,38.0,2,0
2,bang_bon,2026,3,99,2026-01-12,67.0,38.0,52.5,67.0,3,1
3,bang_bon,2026,4,81,2026-01-19,99.0,67.0,83.0,99.0,4,1
4,bang_bon,2026,5,77,2026-01-26,81.0,99.0,90.0,99.0,5,1


In [184]:
# --- FEATURES / TARGET ---
features = ["lag_1", "lag_2", "rolling_avg_2", "rolling_max_2", "week_of_year"]
target = "label"

X_train = train_set[features]
y_train = train_set[target]

X_test = test_set[features]
y_test = test_set[target]
X_train.head()

,lag_1,lag_2,rolling_avg_2,rolling_max_2,week_of_year
0,0.0,0.0,NaN,NaN,1
1,38.0,0.0,38.0,38.0,2
2,67.0,38.0,52.5,67.0,3
3,99.0,67.0,83.0,99.0,4
4,81.0,99.0,90.0,99.0,5


In [185]:
# --- MODEL ---
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# --- EVALUATION ---
y_pred = model.predict(X_test)

print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
X_test['pred'] = y_pred

# --- PREDICTION (LATEST PER DISTRICT) ---
# latest_rows = monthly_cases.sort_values("date").groupby("district").tail(1)
latest_rows = weekly_cases.sort_values("date").groupby("district").tail(1)

latest_rows["prediction"] = model.predict(latest_rows[features])

# Flag districts
flagged = latest_rows[latest_rows["prediction"] == 1]

print("\nDistricts needing diarrhea medicine:")
print(flagged[["district", "prediction"]])

Precision: 0.8787878787878788
Recall: 0.8529411764705882
F1 Score: 0.8656716417910447
Confusion Matrix:
 [[98  4]
 [ 5 29]]
              precision    recall  f1-score   support

           0       0.95      0.96      0.96       102
           1       0.88      0.85      0.87        34

    accuracy                           0.93       136
   macro avg       0.92      0.91      0.91       136
weighted avg       0.93      0.93      0.93       136


Districts needing diarrhea medicine:
       district  prediction
488      prawet           1
661    watthana           1
580  suan_luang           1


C:\Users\thanyathorn\AppData\Local\Temp\ipykernel_18456\2956063081.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test['pred'] = y_pred


In [186]:
print(monthly_cases["month"].unique())      # what months exist
print(monthly_cases["year"].unique())       # what years exist
print(monthly_cases["date"].min(), monthly_cases["date"].max())  # full date range

[3]
[2026]
2026-03-01 00:00:00 2026-03-01 00:00:00


In [187]:
X_test.head(20)

,lag_1,lag_2,rolling_avg_2,rolling_max_2,week_of_year,pred
540,5.0,1.0,3.0,5.0,12,0
541,6.0,5.0,5.5,6.0,13,0
542,0.0,0.0,NaN,NaN,1,0
543,50.0,0.0,50.0,50.0,2,1
544,76.0,50.0,63.0,76.0,3,1
545,88.0,76.0,82.0,88.0,4,1
546,86.0,88.0,87.0,88.0,5,1
547,84.0,86.0,85.0,86.0,6,1
548,71.0,84.0,77.5,84.0,7,0
549,49.0,71.0,60.0,71.0,8,0


In [189]:
# X_test['month_of_year'].value_counts()
X_test['week_of_year'].value_counts()

week_of_year
12    11
13    11
1     10
2     10
3     10
4     10
5     10
6     10
7     10
8     10
9     10
10    10
11    10
14     4
Name: count, dtype: Int64